In [1]:
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

from transformers import AutoTokenizer

/home/sonuts/pytorch_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
file_path = "data.md"

with open(file_path, "r", encoding = "utf_8") as f:
    data = f.read()

In [3]:
print(type(data))
print(len(data))

<class 'str'>
28755


In [4]:
markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on = [
    ("#", "Header_1"),
    ("##", "Header_2"),
    ("###", "Header_3")
])

chunks = markdown_splitter.split_text(data)

In [5]:
chunks

[Document(metadata={'Header_1': 'Blood Bank Operations', 'Header_2': 'SOP Information'}, page_content='Department: Blood Bank Operations\nSOP Code: BB-SOP-001\nVersion: 1.0\nEffective Date: 01-Apr-2026'),
 Document(metadata={'Header_1': 'Blood Bank Operations', 'Header_2': 'Purpose'}, page_content='To define standardized procedures for the safe collection, testing, storage, and transfusion of blood and blood components in order to ensure patient safety and regulatory compliance.'),
 Document(metadata={'Header_1': 'Blood Bank Operations', 'Header_2': 'Scope'}, page_content='This SOP applies to all personnel involved in blood bank operations, including Blood Bank Officers, laboratory technicians, nursing staff, and duty doctors. It covers donor management, blood processing, storage, issue, and transfusion support.'),
 Document(metadata={'Header_1': 'Blood Bank Operations', 'Header_2': 'Procedure', 'Header_3': 'Donor Registration and Screening'}, page_content='1. Register donor details in

In [6]:
print(len(chunks))

131


In [7]:
tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")

def token_len(text):
    return len(tokenizer.encode(text, add_special_tokens = False))

recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 250,
    chunk_overlap = 60,
    length_function = token_len,
    separators = ["\n\n", "\n", ".", " "]
)

final_chunks = recursive_splitter.split_documents(chunks)

In [8]:
final_chunks

[Document(metadata={'Header_1': 'Blood Bank Operations', 'Header_2': 'SOP Information'}, page_content='Department: Blood Bank Operations\nSOP Code: BB-SOP-001\nVersion: 1.0\nEffective Date: 01-Apr-2026'),
 Document(metadata={'Header_1': 'Blood Bank Operations', 'Header_2': 'Purpose'}, page_content='To define standardized procedures for the safe collection, testing, storage, and transfusion of blood and blood components in order to ensure patient safety and regulatory compliance.'),
 Document(metadata={'Header_1': 'Blood Bank Operations', 'Header_2': 'Scope'}, page_content='This SOP applies to all personnel involved in blood bank operations, including Blood Bank Officers, laboratory technicians, nursing staff, and duty doctors. It covers donor management, blood processing, storage, issue, and transfusion support.'),
 Document(metadata={'Header_1': 'Blood Bank Operations', 'Header_2': 'Procedure', 'Header_3': 'Donor Registration and Screening'}, page_content='1. Register donor details in

In [9]:
print(len(final_chunks))

131


In [10]:
embeddings = HuggingFaceEmbeddings(
    model_name = "sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs = {"device": "cuda"},
    encode_kwargs = {"normalize_embeddings": True}
)

In [11]:
vector_db = FAISS.from_documents(final_chunks, embeddings)

In [12]:
vector_db.save_local("faiss_index")